# spellchk: default program

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read the scores.csv file
df = pd.read_csv('../output/dev_score_details.csv')

### Anlysis on Edit Distance

In [6]:
# Distribution of edit distances
print(f"\nEdit Distance Distribution:")
edit_dist_counts = df['Edit_Distance'].value_counts().sort_index()
for dist, count in edit_dist_counts.items():
    print(f"  Distance {dist}: {count} occurrences")


Edit Distance Distribution:
  Distance 1: 1200 occurrences
  Distance 2: 403 occurrences
  Distance 3: 88 occurrences
  Distance 4: 24 occurrences
  Distance 5: 7 occurrences
  Distance 6: 1 occurrences
  Distance 7: 1 occurrences


### Analysis on Transformer Score

In [5]:
# Number of N/A in Transformer Score
num_na = df['Transformer_Score'].isna().sum()
total_entries = len(df)
na_percentage = (num_na / total_entries) * 100
print(f"\n\nTRANSFORMER SCORE N/A COUNT: {num_na} out of {total_entries} ({na_percentage:.2f}%)")

# How many ground truths were found in top 10, top 20, etc.
print(f"\nGround Truth Rank Distribution:")
if df['Rank_Numeric'].notna().any():
    in_top_5 = (df['Rank_Numeric'] <= 5).sum()
    in_top_10 = (df['Rank_Numeric'] <= 10).sum()
    in_top_20 = (df['Rank_Numeric'] <= 20).sum()
    in_top_50 = (df['Rank_Numeric'] <= 50).sum()
    in_top_100 = (df['Rank_Numeric'] <= 100).sum()
    
    print(f"  In Top 5: {in_top_5} ({in_top_5/total_entries*100:.2f}%)")
    print(f"  In Top 10: {in_top_10} ({in_top_10/total_entries*100:.2f}%)")
    print(f"  In Top 20: {in_top_20} ({in_top_20/total_entries*100:.2f}%)")
    print(f"  In Top 50: {in_top_50} ({in_top_50/total_entries*100:.2f}%)")
    print(f"  In Top 100: {in_top_100} ({in_top_100/total_entries*100:.2f}%)")
else:
    print(f"  Not Found (N/A): {num_na} ({na_percentage:.2f}%)")




TRANSFORMER SCORE N/A COUNT: 477 out of 1724 (27.67%)

Ground Truth Rank Distribution:
  In Top 5: 766 (44.43%)
  In Top 10: 907 (52.61%)
  In Top 20: 1020 (59.16%)
  In Top 50: 1156 (67.05%)
  In Top 100: 1247 (72.33%)


In [9]:
# ============================================================================
# ANALYSIS 1: All Incorrect Predictions
# ============================================================================
print("\n" + "="*80)
print("INCORRECT PREDICTIONS")
print("="*80)

incorrect_df = df[df['Selected_Word'].str.lower() != df['Ground_Truth'].str.lower()]
num_incorrect = len(incorrect_df)
incorrect_percentage = (num_incorrect / total_entries) * 100

print(f"\nTotal Incorrect: {num_incorrect} out of {total_entries} ({incorrect_percentage:.2f}%)")
print(f"Total Correct: {total_entries - num_incorrect} ({100 - incorrect_percentage:.2f}%)")

if num_incorrect > 0:
    print("\n" + incorrect_df[['Typo', 'Ground_Truth', 'Selected_Word', 'Transformer_Score', 
                                'Rank', 'Edit_Distance', 'Similarity_Score']].to_string())


INCORRECT PREDICTIONS

Total Incorrect: 518 out of 1724 (30.05%)
Total Correct: 1206 (69.95%)

                         Typo              Ground_Truth    Selected_Word  Transformer_Score  Rank  Edit_Distance  Similarity_Score
1                      didm't                    didn't              did                NaN   NaN              1          0.833333
10                       Rjel                    Rachel               He                NaN   NaN              3          0.600000
11                  uhclaspig                unclasping         clapping                NaN   NaN              2          0.842105
15                  conventnn                convention          consent                NaN   NaN              2          0.842105
18                    vnished                  vanished           raised                NaN   NaN              1          0.933333
28                   imprtont                 important         implicit                NaN   NaN              2      

In [10]:
# ============================================================================
# ANALYSIS 2: Incorrect Predictions where Ground Truth was Available
# ============================================================================
print("\n\n" + "="*80)
print("INCORRECT PREDICTIONS (Ground Truth WAS in Top-100)")
print("="*80)

incorrect_with_score_df = df[
    (df['Selected_Word'].str.lower() != df['Ground_Truth'].str.lower()) & 
    (df['Transformer_Score'].notna())
]
num_incorrect_with_score = len(incorrect_with_score_df)
incorrect_with_score_percentage = (num_incorrect_with_score / total_entries) * 100

print(f"\nIncorrect (ground truth available): {num_incorrect_with_score} out of {total_entries} ({incorrect_with_score_percentage:.2f}%)")
print(f"Incorrect (ground truth NOT available): {num_incorrect - num_incorrect_with_score}")

if num_incorrect_with_score > 0:
    print("\n" + incorrect_with_score_df[['Typo', 'Ground_Truth', 'Selected_Word', 
                                          'Transformer_Score', 'Rank', 'Edit_Distance', 
                                          'Similarity_Score']].to_string())
    
    print(f"\n\nStatistics:")
    print(f"  Avg Rank of Ground Truth: {incorrect_with_score_df['Rank_Numeric'].mean():.2f}")
    print(f"  Avg Edit Distance: {incorrect_with_score_df['Edit_Distance'].mean():.2f}")
    print(f"  Avg Similarity Score: {incorrect_with_score_df['Similarity_Score'].astype(float).mean():.4f}")



INCORRECT PREDICTIONS (Ground Truth WAS in Top-100)

Incorrect (ground truth available): 41 out of 1724 (2.38%)
Incorrect (ground truth NOT available): 477

           Typo Ground_Truth Selected_Word  Transformer_Score  Rank  Edit_Distance  Similarity_Score
63        Theee        These         There           0.068274   3.0              1          0.800000
73           ad          and            as           0.002769  31.0              1          0.800000
75          col         cool          cold           0.003976  40.0              1          0.857143
138         ffm         from           for           0.105097   4.0              2          0.571429
246   crrntrues    centuries     countries           0.341597   2.0              4          0.666667
260           s           is             s           0.082115   1.0              1          0.666667
330     appeard     appeared        appear           0.048655   3.0              1          0.933333
454       the4e        there     